# StreamGuard — XGBoost Magic Features
## Final 70 / 15 / 15 Chronological Evaluation

This notebook is the final evaluation version.

```text
Oldest 70%  -> TRAIN
Next 15%    -> VALIDATION
Newest 15%  -> FINAL TEST
```

Rules:
- Fit all frequency/category/group mappings on TRAIN only.
- Use VALIDATION to choose the decision threshold.
- Do not use TEST until final scoring.
- Do not tune after looking at TEST.


## 1. Install dependencies


In [ ]:
%pip install -q xgboost joblib


## 2. Imports


In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


## 3. Paths


In [ ]:
DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRANSACTION_FILE = DATA_DIR / "train_transaction.csv"
IDENTITY_FILE = DATA_DIR / "train_identity.csv"


## 4. Load and merge


In [ ]:
transaction = pd.read_csv(TRANSACTION_FILE)
identity = pd.read_csv(IDENTITY_FILE)

df = transaction.merge(identity, on="TransactionID", how="left")
df = df.sort_values("TransactionDT").reset_index(drop=True)

print("Merged shape:", df.shape)
print("Overall fraud rate:", df["isFraud"].mean())


## 5. Selected V features


In [ ]:
v_numbers = [
    1,3,4,6,8,11,13,14,17,20,23,26,27,30,
    36,37,40,41,44,47,48,54,56,59,62,65,67,68,70,
    76,78,80,82,86,88,89,91,
    107,108,111,115,117,120,121,123,124,127,129,130,136,
    138,139,142,147,156,162,165,160,166,178,176,173,182,
    187,203,205,207,215,169,171,175,180,185,188,198,210,209,
    218,223,224,226,228,229,235,240,258,257,253,252,260,261,
    264,266,267,274,277,220,221,234,238,250,271,
    294,284,285,286,291,297,303,305,307,309,310,320,
    281,283,289,296,301,314,
]

selected_v_cols = [f"V{i}" for i in v_numbers if f"V{i}" in df.columns]
print("Selected V features:", len(selected_v_cols))


## 6. Keep useful raw columns


In [ ]:
base_cols = [
    "TransactionID","TransactionDT","TransactionAmt","isFraud",
    "ProductCD","card1","card2","card3","card4","card5","card6",
    "addr1","addr2","dist1","dist2","P_emaildomain","R_emaildomain",
]

base_cols += [f"C{i}" for i in range(1,15) if f"C{i}" in df.columns]
base_cols += [f"D{i}" for i in range(1,16) if f"D{i}" in df.columns]
base_cols += [f"M{i}" for i in range(1,10) if f"M{i}" in df.columns]

identity_cols = [
    col for col in df.columns
    if col.startswith("id_") or col in ["DeviceType","DeviceInfo"]
]

keep_cols = []
seen = set()
for col in base_cols + selected_v_cols + identity_cols:
    if col in df.columns and col not in seen:
        keep_cols.append(col)
        seen.add(col)

df = df[keep_cols].copy()
print("Working shape:", df.shape)


## 7. Row-local features


In [ ]:
df["cents"] = (df["TransactionAmt"] - np.floor(df["TransactionAmt"])).astype("float32")
df["day"] = (df["TransactionDT"] / 86400.0).astype("float32")
df["DT_M"] = (df["TransactionDT"] // (86400 * 30)).astype("int16")
df["tx_hour"] = ((df["TransactionDT"] // 3600) % 24).astype("int8")
df["tx_dow"] = ((df["TransactionDT"] // 86400) % 7).astype("int8")
df["amt_log1p"] = np.log1p(df["TransactionAmt"]).astype("float32")


## 8. Normalize D columns


In [ ]:
for i in range(1,16):
    if i in [1,2,3,5,9]:
        continue

    col = f"D{i}"
    if col in df.columns:
        df[col] = (
            df[col] - df["TransactionDT"] / 86400.0
        ).astype("float32")


## 9. Construct pseudo-customer UID


In [ ]:
df["card1_addr1"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str)

df["card1_addr1_P_emaildomain"] = (
    df["card1_addr1"] + "_" + df["P_emaildomain"].astype(str)
)

df["uid"] = (
    df["card1_addr1"] + "_" +
    np.floor(df["day"] - df["D1"]).astype(str)
)

df[["card1","addr1","D1","day","uid"]].head()


## 10. Final chronological split: 70 / 15 / 15


In [ ]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df.iloc[:train_end].copy()
val = df.iloc[train_end:val_end].copy()
test = df.iloc[val_end:].copy()

print("Total:", n)
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))
print()
print("Train fraud rate:", train["isFraud"].mean())
print("Validation fraud rate:", val["isFraud"].mean())
print("Test fraud rate:", test["isFraud"].mean())


## 11. Frequency encoding


In [ ]:
def fit_frequency_map(series):
    s = (
        series.astype("object")
        .where(series.notna(), "__MISSING__")
        .astype(str)
    )
    return s.value_counts(normalize=True, dropna=False).to_dict()

def apply_frequency_map(series, mapping):
    s = (
        series.astype("object")
        .where(series.notna(), "__MISSING__")
        .astype(str)
    )
    return s.map(mapping).fillna(0).astype("float32")

frequency_columns = [
    "addr1","card1","card2","card3","P_emaildomain",
    "card1_addr1","card1_addr1_P_emaildomain","uid",
]

frequency_maps = {}

for column in frequency_columns:
    if column not in train.columns:
        continue

    mapping = fit_frequency_map(train[column])
    frequency_maps[column] = mapping

    for frame in [train, val, test]:
        frame[f"{column}_FE"] = apply_frequency_map(frame[column], mapping)

print("Frequency maps:", len(frequency_maps))


## 12. Train-only group mean/std helper


In [ ]:
group_stat_maps = {}

def add_group_mean_std(train_df, val_df, test_df, group_col, value_col):
    stats = (
        train_df
        .groupby(group_col)[value_col]
        .agg(["mean","std"])
    )

    mean_map = stats["mean"].to_dict()
    std_map = stats["std"].to_dict()

    group_stat_maps[(group_col,value_col,"mean")] = mean_map
    group_stat_maps[(group_col,value_col,"std")] = std_map

    mean_name = f"{value_col}_{group_col}_mean"
    std_name = f"{value_col}_{group_col}_std"

    for frame in [train_df,val_df,test_df]:
        frame[mean_name] = frame[group_col].map(mean_map).astype("float32")
        frame[std_name] = frame[group_col].map(std_map).astype("float32")


## 13. Card/address aggregates


In [ ]:
group_columns = ["card1","card1_addr1","card1_addr1_P_emaildomain"]
aggregate_values = ["TransactionAmt","D9","D11"]

for group_col in group_columns:
    for value_col in aggregate_values:
        if group_col in train.columns and value_col in train.columns:
            add_group_mean_std(
                train,val,test,
                group_col,
                value_col
            )


## 14. UID behavioural mean/std features


In [ ]:
for value_col in ["TransactionAmt","D4","D9","D10","D15"]:
    if value_col in train.columns:
        add_group_mean_std(
            train,val,test,
            "uid",
            value_col
        )


## 15. UID means for C features


In [ ]:
uid_mean_maps = {}

for value_col in [f"C{i}" for i in range(1,15) if i != 3]:
    if value_col not in train.columns:
        continue

    mapping = train.groupby("uid")[value_col].mean().to_dict()
    uid_mean_maps[value_col] = mapping

    feature_name = f"{value_col}_uid_mean"

    for frame in [train,val,test]:
        frame[feature_name] = frame["uid"].map(mapping).astype("float32")


## 16. Numeric M features and UID means


In [ ]:
m_numeric_cols = []

for i in range(1,10):
    col = f"M{i}"

    if col not in train.columns:
        continue

    numeric_col = f"{col}_numeric"

    tf_map = {
        "T":1.0,
        "F":0.0,
        "True":1.0,
        "False":0.0,
    }

    for frame in [train,val,test]:
        frame[numeric_col] = (
            frame[col]
            .astype(str)
            .map(tf_map)
            .astype("float32")
        )

    m_numeric_cols.append(numeric_col)

uid_m_mean_maps = {}

for value_col in m_numeric_cols:
    mapping = train.groupby("uid")[value_col].mean().to_dict()
    uid_m_mean_maps[value_col] = mapping

    feature_name = f"{value_col}_uid_mean"

    for frame in [train,val,test]:
        frame[feature_name] = frame["uid"].map(mapping).astype("float32")


## 17. UID unique-count behavioural features


In [ ]:
uid_nunique_maps = {}

uid_nunique_columns = [
    "P_emaildomain","dist1","DT_M","id_02","cents",
    "C13","V314","V127","V136","V309","V307","V320",
]

for value_col in uid_nunique_columns:
    if value_col not in train.columns:
        continue

    mapping = train.groupby("uid")[value_col].nunique().to_dict()
    uid_nunique_maps[value_col] = mapping

    feature_name = f"uid_{value_col}_ct"

    for frame in [train,val,test]:
        frame[feature_name] = (
            frame["uid"]
            .map(mapping)
            .fillna(0)
            .astype("float32")
        )


## 18. outsider15


In [ ]:
if "D1" in train.columns and "D15" in train.columns:
    for frame in [train,val,test]:
        frame["outsider15"] = (
            np.abs(frame["D1"] - frame["D15"]) > 3
        ).astype("int8")


## 19. Train-only categorical encoding


In [ ]:
category_maps = {}

categorical_columns = (
    train
    .select_dtypes(include=["object","category","string"])
    .columns
    .tolist()
)

for column in categorical_columns:

    if column == "uid":
        continue

    train_values = (
        train[column]
        .astype("object")
        .where(train[column].notna(), "__MISSING__")
        .astype(str)
    )

    values = train_values.drop_duplicates().tolist()
    mapping = {value:index for index,value in enumerate(values)}

    category_maps[column] = mapping

    for frame in [train,val,test]:

        current = (
            frame[column]
            .astype("object")
            .where(frame[column].notna(), "__MISSING__")
            .astype(str)
        )

        frame[column] = (
            current
            .map(mapping)
            .fillna(-1)
            .astype("int32")
        )

print("Categorical maps:", len(category_maps))


## 20. Final feature list


In [ ]:
exclude = {
    "isFraud",
    "TransactionID",
    "TransactionDT",
    "uid",
    "day",
}

features = [
    column
    for column in train.columns
    if column not in exclude
    and pd.api.types.is_numeric_dtype(train[column])
]

print("Final feature count:", len(features))
print("Train matrix:", train[features].shape)
print("Validation matrix:", val[features].shape)
print("Test matrix:", test[features].shape)


## 21. XGBoost model


In [ ]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=2000,
    max_depth=12,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.4,
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

print("XGBoost version:", xgb.__version__)

# Optional GPU:
# model.set_params(device="cuda")


## 22. Train on 70% only


In [ ]:
model.fit(
    train[features],
    train["isFraud"],
    eval_set=[
        (
            val[features],
            val["isFraud"],
        )
    ],
    verbose=50,
)


## 23. Validation metrics


In [ ]:
val_probs = model.predict_proba(val[features])[:,1]

val_pr_auc = average_precision_score(
    val["isFraud"],
    val_probs
)

val_roc_auc = roc_auc_score(
    val["isFraud"],
    val_probs
)

print("="*60)
print("VALIDATION RESULTS")
print("="*60)
print("Validation PR-AUC:", val_pr_auc)
print("Validation ROC-AUC:", val_roc_auc)


## 24. Choose threshold on validation only


In [ ]:
best_threshold = None
best_val_f1 = -1

for threshold in np.arange(0.01,1.00,0.01):

    pred = (val_probs >= threshold).astype(int)

    score = f1_score(
        val["isFraud"],
        pred,
        zero_division=0,
    )

    if score > best_val_f1:
        best_val_f1 = score
        best_threshold = float(threshold)

print("Chosen threshold:", best_threshold)
print("Validation F1:", best_val_f1)


# 25. FINAL untouched test evaluation

This is the final generalisation score.

Do not use these results to tune the model again.


In [ ]:
test_probs = model.predict_proba(test[features])[:,1]

test_pr_auc = average_precision_score(
    test["isFraud"],
    test_probs
)

test_roc_auc = roc_auc_score(
    test["isFraud"],
    test_probs
)

test_pred = (test_probs >= best_threshold).astype(int)

print("="*60)
print("FINAL UNTOUCHED TEST RESULTS")
print("="*60)
print("Test PR-AUC:", test_pr_auc)
print("Test ROC-AUC:", test_roc_auc)

print()
print(
    classification_report(
        test["isFraud"],
        test_pred,
        digits=4,
    )
)

cm = confusion_matrix(
    test["isFraud"],
    test_pred
)

print("Confusion matrix:")
print(cm)

tn,fp,fn,tp = cm.ravel()

print()
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print()
print(
    "Precision:",
    precision_score(
        test["isFraud"],
        test_pred,
        zero_division=0,
    )
)

print(
    "Recall:",
    recall_score(
        test["isFraud"],
        test_pred,
        zero_division=0,
    )
)

print(
    "F1:",
    f1_score(
        test["isFraud"],
        test_pred,
        zero_division=0,
    )
)


## 26. Validation vs test summary


In [ ]:
results = pd.DataFrame({
    "split":["validation","final_test"],
    "pr_auc":[val_pr_auc,test_pr_auc],
    "roc_auc":[val_roc_auc,test_roc_auc],
})

results


## 27. Feature importance


In [ ]:
importance = pd.DataFrame({
    "feature":features,
    "importance":model.feature_importances_,
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

importance.head(30)


## 28. Save evaluation artifacts


In [ ]:
EVAL_MODEL_PATH = MODEL_DIR / "streamguard_xgb_magic_70pct_eval_model.joblib"
EVAL_ARTIFACT_PATH = MODEL_DIR / "streamguard_xgb_magic_70_15_15_eval_artifacts.joblib"

joblib.dump(
    model,
    EVAL_MODEL_PATH
)

evaluation_artifacts = {
    "split":{
        "train":0.70,
        "validation":0.15,
        "test":0.15,
    },

    "features":features,
    "selected_v_cols":selected_v_cols,
    "best_threshold":best_threshold,

    "validation_pr_auc":val_pr_auc,
    "validation_roc_auc":val_roc_auc,
    "validation_f1":best_val_f1,

    "test_pr_auc":test_pr_auc,
    "test_roc_auc":test_roc_auc,

    "frequency_maps":frequency_maps,
    "group_stat_maps":group_stat_maps,
    "uid_mean_maps":uid_mean_maps,
    "uid_m_mean_maps":uid_m_mean_maps,
    "uid_nunique_maps":uid_nunique_maps,
    "category_maps":category_maps,
}

joblib.dump(
    evaluation_artifacts,
    EVAL_ARTIFACT_PATH
)

print("Saved:", EVAL_MODEL_PATH.resolve())
print("Saved:", EVAL_ARTIFACT_PATH.resolve())


# 29. What to send me

After it finishes, send:

```text
Validation PR-AUC:
Validation ROC-AUC:
Chosen threshold:

Test PR-AUC:
Test ROC-AUC:
Precision:
Recall:
F1:
Confusion matrix:
```

If the final test result is acceptable, the next step is to rebuild preprocessing on 100% of the labelled data and train the final deployment model for Kafka.
